In [ ]:
# =============================================================================
# 0. INSTALL DEPENDENCIES (Kaggle)
# =============================================================================
!apt-get update -qq
!apt-get install -y -qq openbabel autodock-vina
!pip install -q biotite meeko openmm rdkit scipy seaborn matplotlib gemmi

print('=== Dependency installation complete ===')

# Verify imports immediately so we fail fast if something is missing
import importlib
for pkg in ['biotite', 'meeko', 'openmm', 'rdkit', 'scipy', 'seaborn', 'matplotlib', 'gemmi']:
    importlib.import_module(pkg)
    print(f'  {pkg}: OK')

In [ ]:
# =============================================================================
# 1. PATH SETUP FOR KAGGLE
# =============================================================================
# CodeOcean uses /data and /results. Kaggle uses /kaggle/working.

import os

WORKING_DIR = "/kaggle/working"
DATA_DIR = os.path.join(WORKING_DIR, "data")
RESULTS_DIR = os.path.join(WORKING_DIR, "results")

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"WORKING_DIR: {WORKING_DIR}")
print(f"DATA_DIR:    {DATA_DIR}")
print(f"RESULTS_DIR: {RESULTS_DIR}")

In [ ]:
# =============================================================================
# 2. GPR151 CONTEXTUALIZATION DASHBOARD
# =============================================================================

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist
import biotite.structure.io as strucio
import biotite.structure as struc
import biotite.database.afdb as afdb

print("Downloading GPR151 structure via Biotite AFDB (AlphaFold ID: Q8TDV0)...")
cif_filename = afdb.fetch("Q8TDV0", "cif", target_path=DATA_DIR)

array = strucio.load_structure(cif_filename)
protein = array[struc.filter_amino_acids(array)]

res_ids = np.unique(protein.res_id)
calphas = protein[protein.atom_name == "CA"]

fig, axs = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("GPR151 Contextualization Dashboard", fontsize=18)

# Plot 1: Mock Sequence Conservation
mock_entropy = np.random.uniform(0.1, 2.5, size=len(res_ids))
axs[0, 0].plot(res_ids, mock_entropy, color='purple', alpha=0.7)
axs[0, 0].set_title("1. Sequence Conservation (Mock Entropy)")
axs[0, 0].set_xlabel("Residue Number")
axs[0, 0].set_ylabel("Shannon Entropy (Bits)")

# Plot 2: Per-Residue SASA
print("Calculating Solvent Accessible Surface Area...")
atom_sasa = struc.sasa(protein)
res_sasa = np.array([np.sum(atom_sasa[protein.res_id == r]) for r in res_ids])
axs[0, 1].plot(res_ids, res_sasa, color='teal')
axs[0, 1].set_title("2. Per-Residue SASA")
axs[0, 1].set_xlabel("Residue Number")
axs[0, 1].set_ylabel("SASA (Å²)")

# Plot 3: C-alpha Contact Map
print("Generating Distance Matrix...")
coords = calphas.coord
dist_matrix = cdist(coords, coords, metric='euclidean')
sns.heatmap(dist_matrix, ax=axs[1, 0], cmap='viridis_r',
            cbar_kws={'label': 'Distance (Å)'},
            xticklabels=50, yticklabels=50)
axs[1, 0].set_title("3. C-alpha Contact Map")
axs[1, 0].set_xlabel("Residue Index")
axs[1, 0].set_ylabel("Residue Index")

# Plot 4: Mock Interaction Energy
mock_energy = np.random.normal(-1, 2, size=len(res_ids))
mock_energy[mock_energy > 0] = 0
axs[1, 1].bar(res_ids, mock_energy, color='crimson')
axs[1, 1].set_title("4. Interaction Energy (Mock Data)")
axs[1, 1].set_xlabel("Residue Number")
axs[1, 1].set_ylabel("Energy Contribution (kcal/mol)")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plot_path = os.path.join(RESULTS_DIR, "GPR151_contextualization_dashboard.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"Saved plot to {plot_path}")
plt.show()

In [ ]:
# =============================================================================
# 3. FETCH & CLEAN RECEPTOR (GPR151)
# =============================================================================

import os
import subprocess
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cdist
import biotite.structure.io as strucio
import biotite.structure as struc
import biotite.database.afdb as afdb
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation

LIGAND_SMILES = "COC1=CC=CC=C1CNC(=O)CC2=C(C)OC3=C(C2=O)C=C4C(=C3C)OC=C4"
LIGAND_RES_NAME = "UNL"

print("Downloading GPR151...")
cif_filename = afdb.fetch("Q8TDV0", "cif", target_path=DATA_DIR)
protein = strucio.load_structure(cif_filename)
protein = protein[struc.filter_amino_acids(protein)]

protein_clean_path = os.path.join(RESULTS_DIR, "GPR151_clean.pdb")
strucio.save_structure(protein_clean_path, protein)
print(f"Saved cleaned protein to: {protein_clean_path}")
print(f"Residues: {len(np.unique(protein.res_id))}")

In [ ]:
# =============================================================================
# 4. ENERGY MINIMIZATION (OPTIONAL) & RECEPTOR PREP
# =============================================================================
# WARNING: OpenMM relaxation on CPU can take 5-15 minutes.
# If you want to skip it, set SKIP_RELAXATION = True and the notebook will use
# the cleaned PDB directly.

SKIP_RELAXATION = False  # Set to True to skip the slow minimization step

relaxed_path = os.path.join(RESULTS_DIR, "GPR151_relaxed.pdb")
prepared_pdbqt = os.path.join(RESULTS_DIR, "GPR151_prepared.pdbqt")

if not SKIP_RELAXATION:
    print("Relaxing protein structure with OpenMM (this will take several minutes)...")
    from openmm.app import PDBFile, Modeller, ForceField, Simulation, PME, HBonds
    from openmm import LangevinIntegrator
    from openmm.unit import kelvin, picosecond, picoseconds, nanometer, molar

    pdb = PDBFile(protein_clean_path)
    forcefield = ForceField('amber14-all.xml', 'amber14/tip3p.xml')
    modeller = Modeller(pdb.topology, pdb.positions)

    print("Adding Hydrogens...")
    modeller.addHydrogens(forcefield)

    print("Adding solvent box (0.5 nm padding)...")
    modeller.addSolvent(forcefield, padding=0.5*nanometer, ionicStrength=0.15*molar)

    system = forcefield.createSystem(modeller.topology, nonbondedMethod=PME,
                                     nonbondedCutoff=1*nanometer, constraints=HBonds)

    integrator = LangevinIntegrator(300*kelvin, 1/picosecond, 0.002*picoseconds)
    simulation = Simulation(modeller.topology, system, integrator)
    simulation.context.setPositions(modeller.positions)

    print("Minimizing energy...")
    simulation.minimizeEnergy()

    positions = simulation.context.getState(getPositions=True).getPositions()
    with open(relaxed_path, 'w') as f:
        PDBFile.writeFile(simulation.topology, positions, f)
    print(f"Relaxation complete. Saved to: {relaxed_path}")
else:
    print("Skipping OpenMM relaxation. Using cleaned PDB directly.")
    relaxed_path = protein_clean_path

# --- Prepare receptor for Vina with Open Babel ---
print("\nConverting relaxed structure to PDBQT with Open Babel...")
cmd = f"obabel {relaxed_path} -O {prepared_pdbqt} -xr -p 7.4"
print(f"Running: {cmd}")
ret = os.system(cmd)

if ret != 0:
    raise RuntimeError(f"Open Babel failed with exit code {ret}. Is openbabel installed?")

if not os.path.exists(prepared_pdbqt) or os.path.getsize(prepared_pdbqt) == 0:
    raise RuntimeError("Receptor PDBQT file is empty. Check Open Babel output above.")

print(f"SUCCESS: Receptor prepared -> {prepared_pdbqt}")
print(f"File size: {os.path.getsize(prepared_pdbqt)} bytes")

In [ ]:
# =============================================================================
# 5. LIGAND PREPARATION (GUM3)
# =============================================================================

print("Preparing Ligand (GUM3)...")
mol = Chem.MolFromSmiles(LIGAND_SMILES)
if mol is None:
    raise ValueError(f"Failed to parse SMILES: {LIGAND_SMILES}")

mol = Chem.AddHs(mol)
AllChem.EmbedMolecule(mol, randomSeed=42)
AllChem.MMFFOptimizeMolecule(mol)

prep = MoleculePreparation()
prep.prepare(mol)

ligand_pdbqt = os.path.join(RESULTS_DIR, "GUM3.pdbqt")
with open(ligand_pdbqt, "w") as f:
    f.write(prep.write_pdbqt_string())

if not os.path.exists(ligand_pdbqt) or os.path.getsize(ligand_pdbqt) == 0:
    raise RuntimeError("Ligand PDBQT file is empty.")

print(f"SUCCESS: Ligand prepared -> {ligand_pdbqt}")
print(f"File size: {os.path.getsize(ligand_pdbqt)} bytes")

In [ ]:
# =============================================================================
# 6. MOLECULAR DOCKING WITH AUTODOCK VINA
# =============================================================================
# NOTE: Exhaustiveness=8 is fast for testing. For publication-quality results,
# change to 32 or 64.

docked_pdbqt = os.path.join(RESULTS_DIR, "GUM3_docked.pdbqt")
vina_log = os.path.join(RESULTS_DIR, "vina_log.txt")

print("Running Vina Docking...")
cmd = [
    "vina",
    "--receptor", prepared_pdbqt,
    "--ligand", ligand_pdbqt,
    "--center_x", "-5.350",
    "--center_y", "2.548",
    "--center_z", "1.510",
    "--size_x", "15",
    "--size_y", "15",
    "--size_z", "15",
    "--exhaustiveness", "8",
    "--out", docked_pdbqt
]

result = subprocess.run(cmd, capture_output=True, text=True)

# Save full log
with open(vina_log, "w") as f:
    f.write(result.stdout)
    f.write(result.stderr)

print(result.stdout)
if result.returncode != 0:
    print(f"ERROR: Vina exited with code {result.returncode}")
    print(result.stderr)
else:
    print(f"\nSUCCESS: Docking complete -> {docked_pdbqt}")
    print(f"Log saved to -> {vina_log}")

    # Parse binding affinities from the log
    print("\n--- Binding Affinities ---")
    for line in result.stdout.splitlines():
        if line.strip().startswith("1 ") or line.strip().startswith("2 ") or line.strip().startswith("3 "):
            print(line.strip())
        elif "kcal/mol" in line and "-----" not in line:
            print(line.strip())

if not os.path.exists(docked_pdbqt) or os.path.getsize(docked_pdbqt) == 0:
    raise RuntimeError("Docked output file is empty. Check Vina log above.")

In [ ]:
# =============================================================================
# 7. POST-DOCKING ANALYSIS & VISUALIZATION
# =============================================================================

print("Loading results for analysis...")
protein = strucio.load_structure(relaxed_path)

# Convert docked ligand to PDB for easier parsing
docked_pdb = os.path.join(RESULTS_DIR, "GUM3_docked.pdb")
ret = os.system(f"obabel {docked_pdbqt} -O {docked_pdb}")
if ret != 0 or not os.path.exists(docked_pdb):
    raise RuntimeError("Failed to convert docked ligand to PDB.")

ligand = strucio.load_structure(docked_pdb)

# Take only the first pose (Vina writes multiple models)
if hasattr(ligand, 'stack_depth') and ligand.stack_depth() > 1:
    ligand_first = ligand.get_stack()[0]
else:
    ligand_first = ligand

lig_coords = ligand_first.coord
res_ids = np.unique(protein.res_id)

fig, axs = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("GPR151 + GUM3 Docking Analysis", fontsize=18)

# 1. Conservation (Mock)
axs[0, 0].plot(res_ids, np.random.uniform(0.1, 2.5, size=len(res_ids)), color='purple')
axs[0, 0].set_title("1. Sequence Conservation (Mock)")
axs[0, 0].set_xlabel("Residue ID")
axs[0, 0].set_ylabel("Entropy Score")

# 2. SASA
atom_sasa = struc.sasa(protein)
res_sasa = np.array([np.sum(atom_sasa[protein.res_id == r]) for r in res_ids])
axs[0, 1].plot(res_ids, res_sasa, color='teal')
axs[0, 1].set_title("2. Solvent Accessible Surface Area")
axs[0, 1].set_xlabel("Residue ID")
axs[0, 1].set_ylabel(r"SASA ($\AA^2$)")

# 3. Ligand-Protein Contact Distances
unique_res_ids = np.unique(protein.res_id)
contact_matrix = np.zeros((len(unique_res_ids), len(lig_coords)))
for i, res_id in enumerate(unique_res_ids):
    res_atoms = protein[protein.res_id == res_id]
    dists = cdist(res_atoms.coord, lig_coords)
    contact_matrix[i, :] = np.min(dists, axis=0)

sns.heatmap(contact_matrix, ax=axs[1, 0], cmap='rocket_r', yticklabels=40)
axs[1, 0].set_title("3. Ligand-Protein Contact Distances")
axs[1, 0].set_xlabel("Ligand Atom Index")
axs[1, 0].set_ylabel("Protein Residue ID")

# 4. Interaction Proxy (Contacts < 4Å)
dist_to_lig = cdist(protein.coord, lig_coords).min(axis=1)
contact_counts = np.array([np.sum(dist_to_lig[protein.res_id == r] < 4.0) for r in res_ids])
axs[1, 1].bar(res_ids, contact_counts, color='orange')
axs[1, 1].set_title("4. Interaction Proxy (Contacts < 4Å)")
axs[1, 1].set_xlabel("Residue ID")
axs[1, 1].set_ylabel("Number of Contacts")

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plot_path = os.path.join(RESULTS_DIR, "GPR151_GUM3_dashboard.png")
plt.savefig(plot_path, dpi=300, bbox_inches='tight')
print(f"Saved plot to: {plot_path}")
plt.show()

print("\n=== ALL DONE ===")
print(f"All outputs saved to: {RESULTS_DIR}")
print("Files generated:")
for f in sorted(os.listdir(RESULTS_DIR)):
    print(f"  - {f}")